# Normalizing flow sample example


In [ ]:
import sys
import os
if ".." not in sys.path:
    sys.path.insert(0, os.path.abspath(".."))

# Add path to more easily load checkpoints of different architectures
model_utils_path = ""  # TODO: PATH_UPDATE fine-tuning codebase path

import torch
from utils.color.linear_to_srgb_converters import LinearRec709ToAgXBase
from utils.color.tonemapping.agx_looks import AgXPunchyLook

In [ ]:
torch_precision = torch.float32
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
lr = 0.07
n_iter = 280

In [ ]:
global_seed = 3 # Can be None

In [ ]:
from utils.train import train_with_criterion
from scenes import SciFiRobotScene
import os

color_space_converter = LinearRec709ToAgXBase(AgXPunchyLook())
scene = SciFiRobotScene(device=device)

from utils.losses.normalizing_flow import create_normalizing_flow_loss

# Example: sample-based unconditional flow loss
# criterion = create_normalizing_flow_loss(
#     flow_checkpoint="",  # TODO: PATH_UPDATE normalizing flow checkpoint
#     flow_dimensionality=64,
#     loss_variant='sample',
#     sample_mode='cosine',
#     num_distinct_samples=1,
#     normalize_sampled_embedding=True,
#     normalize_context=True,
# )

# Example: conditional sample-based flow loss
criterion = create_normalizing_flow_loss(
    flow_checkpoint="",  # TODO: PATH_UPDATE normalizing flow checkpoint
    flow_dimensionality=64,
    conditional_embeddings=torch.Tensor([[0.0, 1.0, 0.0, 0.0, 0.0, 0.0]]),
    use_batch_norm=True,
    loss_variant='sample',
    sample_mode='l2',
    num_distinct_samples=1,
)

output_directory = 'normalizing_flow_sample_example'

train_with_criterion(
    scene,
    lr, n_iter, criterion,
    starting_multiplier_std=(0.4, 0.4, 0.4),
    output_subdirectory_name=output_directory,
    n_results=4,
    torch_precision=torch_precision,
    render_color_space_converter=color_space_converter,
    require_physically_plausible_multipliers=True,
    title_prefix="Sample Normalizing Flow Loss Test",
    device=device,
    save_every=100,
    model_name='sample_normalizing_flow',
    pretrained_source='',
    seed=global_seed
)